In [ ]:
import os
import glob
import math
import numpy as np
from PIL import Image

# Configuration
INPUT_DIR = '../'  # Directory where your .mem files live
OUTPUT_TILEMAP = 'tile_generator_custom/combined_tilemap.png'
TILE_ROWS = 32
TILE_COLS = 32
MAX_TILES = 32

def load_mem_file(filepath):
    """Reads a .mem file and parses the 7-digit binary strings back into a 32x32 array."""
    pixels = []
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            # Convert 7-bit binary string back to integer
            pixel = int(line, 2)
            pixels.append(pixel)
            
    if len(pixels) != TILE_ROWS * TILE_COLS:
        raise ValueError(f"File {filepath} has {len(pixels)} pixels, expected {TILE_ROWS * TILE_COLS}")
        
    return np.array(pixels).reshape((TILE_ROWS, TILE_COLS))

def decode_7bit_to_rgba(tile_data):
    """Reverses the 7-bit packing logic to construct an 8-bit RGBA image."""
    rgba_img = np.zeros((TILE_ROWS, TILE_COLS, 4), dtype=np.uint8)
    
    # Extract bit components using bitwise masking
    a_1bit = (tile_data >> 6) & 0x1
    r_2bit = (tile_data >> 4) & 0x3
    g_2bit = (tile_data >> 2) & 0x3
    b_2bit = tile_data & 0x3
    
    # Upscale 2-bit (0-3) back to 8-bit (0-255). 
    # Multiplying by 85 maps: 0->0, 1->85, 2->170, 3->255 smoothly.
    rgba_img[:, :, 0] = r_2bit * 85  # Red
    rgba_img[:, :, 1] = g_2bit * 85  # Green
    rgba_img[:, :, 2] = b_2bit * 85  # Blue
    
    # Handle alpha (Your encoder rules: 1 = Transparent, 0 = Opaque)
    rgba_img[:, :, 3] = np.where(a_1bit == 1, 0, 255)
    
    return rgba_img

def main():
    # Gather up to 32 .mem files alphabetically/numerically
    mem_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.mem")))[:MAX_TILES]
    num_tiles = len(mem_files)
    
    if num_tiles == 0:
        print(f"Error: No .mem files found in '{INPUT_DIR}'")
        return
        
    print(f"... Found {num_tiles} tile memory files.")
    
    # Calculate grid dimensions dynamically (aiming for a square-ish layout)
    grid_cols = math.ceil(math.sqrt(num_tiles))
    grid_rows = math.ceil(num_tiles / grid_cols)
    
    # Initialize a blank canvas canvas for the entire tilemap sheet
    tilemap_width = grid_cols * TILE_COLS
    tilemap_height = grid_rows * TILE_ROWS
    tilemap_array = np.zeros((tilemap_height, tilemap_width, 4), dtype=np.uint8)
    
    # Stitch the tiles together
    for index, filepath in enumerate(mem_files):
        print(f"Processing [{index+1}/{num_tiles}]: {os.path.basename(filepath)}")
        
        # Load and decode back to RGBA
        tile_data = load_mem_file(filepath)
        rgba_tile = decode_7bit_to_rgba(tile_data)
        
        # Determine coordinate positions on the grid
        tile_row_idx = index // grid_cols
        tile_col_idx = index % grid_cols
        
        y_start = tile_row_idx * TILE_ROWS
        y_end = y_start + TILE_ROWS
        x_start = tile_col_idx * TILE_COLS
        x_end = x_start + TILE_COLS
        
        # Insert tile into the large canvas
        tilemap_array[y_start:y_end, x_start:x_end] = rgba_tile

    # Save the output image map
    print(f"... Saving combined tilemap to: {OUTPUT_TILEMAP}")
    final_image = Image.fromarray(tilemap_array, 'RGBA')
    final_image.save(OUTPUT_TILEMAP)
    print("... Done!")

if __name__ == "__main__":
    main()

... Found 9 tile memory files.
Processing [1/9]: BrickWall2.mem
Processing [2/9]: bartender1.mem
Processing [3/9]: brickWall1.mem
Processing [4/9]: image_data.mem
Processing [5/9]: image_data2.mem
Processing [6/9]: negroni1.mem
Processing [7/9]: negroni2.mem
Processing [8/9]: whiskeyPoster2.mem
Processing [9/9]: whiskeyposter1.mem
... Saving combined tilemap to: tile_generator_custom/combined_tilemap.png
... Done!


C:\Users\noahd\AppData\Local\Temp\ipykernel_24012\1948081651.py:94: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  final_image = Image.fromarray(tilemap_array, 'RGBA')
